# Figure S7 - recurrence, background spread, and timing (true genealogy)

Built from the true simulation genealogies (`run-out.branches`), restricted to *established* origins (clade retains at least `MIN_PROGENY` sampled infections). **A**: origin-count CDF -- recurrence is rare among established lineages (faint = per run, bold = representative). **B**/**C**: for each recurrent substitution, the minimum background AA distance and minimum birth-time gap between its independent origins -- when recurrence does happen, the origins are genetically and temporally far apart. Box-and-strip, pooled over the candidate runs.


## Imports and parameters

In [ ]:
import re
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

%matplotlib inline

HERE = Path.cwd()
sys.path.append(str(HERE.parent / 'scripts'))

from plot_truetree_homoplasy import build_pooled_4panel
from plot_mutation_homoplasy import REPRESENTATIVE_RUN
from antigentools.supplement_style import apply_supplement_style

apply_supplement_style()

BATCH = '2026-07-04-reviewer-runs'
AGG_DIR = HERE.parent / 'results' / 'aggregated' / BATCH
CAND_PATH = HERE.parent / 'data' / BATCH / 'antigen-outputs' / 'candidate_runs.csv'
FIG_DIR = HERE.parent.parent / 'antigen-tex' / 'reviews' / 'round1' / 'figures'
FIG_PREFIX = 'figureS7_truetree_homoplasy'

MIN_PROGENY = 10  # established threshold; the sweep emits every PROGENY_THRESHOLDS value
print(f'restricted to origins with >= {MIN_PROGENY} sampled infections')

## Load the sweep output and restrict to the flu-like candidate runs

In [ ]:
recurrence = pd.read_csv(AGG_DIR / 'truetree_recurrence_by_run.csv')
origin_counts = pd.read_csv(AGG_DIR / 'truetree_origin_counts_by_run.csv')
spread = pd.read_csv(AGG_DIR / 'truetree_origin_spread_by_run.csv')

cand = pd.read_csv(CAND_PATH)


def config_from_path(p):
    m = re.search(r'simulations/([^/]+)/run_', p)
    assert m is not None, f'cannot parse config from candidate path: {p!r}'
    return m.group(1)


cand['config'] = cand['path'].map(config_from_path)
cand_keys = set(zip(cand['config'], cand['run'].astype(int)))


def keep_candidates(df, label):
    keys = list(zip(df['config'], df['run'].astype(int)))
    kept = df[[k in cand_keys for k in keys]].copy()
    assert not kept.empty, f'no candidate runs survived the join for {label}'
    print(f'{label}: {len(kept)} rows from {kept.groupby(["config", "run"]).ngroups} candidate runs')
    return kept


recurrence = keep_candidates(recurrence, 'per-run recurrence')
origin_counts = keep_candidates(origin_counts, 'origin-count ECDF')
spread = keep_candidates(spread, 'origin spread')

for name, df in [('origin_counts', origin_counts), ('spread', spread)]:
    assert MIN_PROGENY in set(df['min_progeny']), \
        f'MIN_PROGENY={MIN_PROGENY} not in {name} thresholds {sorted(set(df["min_progeny"]))}'

## Figure S7

In [ ]:
fig = build_pooled_4panel(origin_counts, spread, REPRESENTATIVE_RUN, min_progeny=MIN_PROGENY)
plt.show()

In [ ]:
FIG_DIR.mkdir(parents=True, exist_ok=True)
for suffix in ('pdf', 'png'):
    path = FIG_DIR / f'{FIG_PREFIX}.{suffix}'
    fig.savefig(path, dpi=300, bbox_inches='tight')
    print(f'Wrote {path}')

## Numbers (pooled across candidate runs)

In [ ]:
est = spread[spread['min_progeny'] == MIN_PROGENY]
for is_epi, name in [(True, 'epitope'), (False, 'non_epitope')]:
    s = est[est['is_epitope'] == is_epi]
    print(f'{name:12s} n={len(s):5d}  '
          f'min background dist AA: median {s["min_background_distance_aa"].median():.1f}  '
          f'min gap yr: median {s["min_birthtime_gap_years"].median():.1f}')